In [1]:
# 08_robustness_leakage.ipynb
#
# Experiment 8 -- Robustness: strong-model leakage and GBM base scorer.
#
# Purpose:
#   (1) Proxy leakage with a gradient-boosted reconstructor (HistGradientBoosting)
#       gives a tighter, higher estimate than logistic leakage, confirming that
#       logistic leakage is a lower bound.
#   (2) The DIR disparity pattern reproduces when the credit scorer is a GBM
#       rather than logistic regression, so the results are not model-specific.
#   Outputs the comparison table and a figure. Grayscale, 600 dpi, PNG+PDF,
#   no captions, no path printing.
#
# Outputs:
#   results/tables/exp_robustness_leakage.csv
#   results/figures/exp8_leakage_logit_vs_gbm.(png|pdf)

import os, warnings, numpy as np, pandas as pd
warnings.filterwarnings("ignore")
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import roc_auc_score
import os
_R=os.path.abspath(os.path.join(os.getcwd(),".."))
if not os.path.isdir(os.path.join(_R,"results")): _R=os.getcwd()
D=os.path.join(_R,"data")

def dir_ratio(y_pred, g):
    ap=(y_pred==0).astype(int); r1=ap[g==1].mean(); r0=ap[g==0].mean()
    return (r1/r0) if r0>0 else np.nan

def leak_logit(X,g):
    Xs=StandardScaler().fit_transform(X.fillna(X.median(numeric_only=True)).values)
    return cross_val_score(LogisticRegression(max_iter=1000),Xs,g,cv=3,scoring="roc_auc").mean()
def leak_gbm(X,g):
    Xf=X.fillna(X.median(numeric_only=True)).values
    return cross_val_score(HistGradientBoostingClassifier(max_iter=200,max_depth=4,random_state=0),
                           Xf,g,cv=3,scoring="roc_auc").mean()

def base_dir(X,y,g,seed,model):
    Xtr,Xte,ytr,yte,gtr,gte=train_test_split(X,y,g,test_size=0.3,random_state=seed,stratify=y)
    if model=="logit":
        sc=StandardScaler().fit(Xtr.fillna(Xtr.median(numeric_only=True)))
        m=LogisticRegression(max_iter=2000).fit(sc.transform(Xtr.fillna(Xtr.median(numeric_only=True))),ytr)
        p=m.predict_proba(sc.transform(Xte.fillna(Xte.median(numeric_only=True))))[:,1]
    else:
        m=HistGradientBoostingClassifier(max_iter=200,max_depth=4,random_state=seed).fit(Xtr,ytr)
        p=m.predict_proba(Xte)[:,1]
    auc=roc_auc_score(yte,p); yp=(p>=0.5).astype(int)
    return auc, dir_ratio(yp,gte)

# loaders reused (drop_protected feature sets)
def german():
    df=pd.read_csv(f"{D}/german_credit.csv"); y=(df.credit_risk==0).astype(int).values; g=(df.age<25).astype(int).values
    cat=[c for c in df.select_dtypes(include=['object','string']).columns]
    X=pd.get_dummies(df.drop(columns=['credit_risk','age']),columns=cat).astype(float); return X,y,g
def taiwan(attr):
    df=pd.read_parquet(f"{D}/taiwan_credit.parquet"); y=df.DEFAULT.astype(int).values
    g=(df.SEX==2).astype(int).values if attr=='sex' else ((df.AGE<30).astype(int).values if attr=='age' else (df.MARRIAGE==2).astype(int).values)
    X=df.drop(columns=[c for c in ['DEFAULT','VIP','SEX_STD','AGE_YEARS','SEX','AGE','MARRIAGE'] if c in df.columns]).astype(float).reset_index(drop=True); return X,y,g
def gmsc():
    df=pd.read_parquet(f"{D}/gmsc_credit.parquet").sample(20000,random_state=0); y=df.DEFAULT.astype(int).values; g=(df.AGE_YEARS<40).astype(int).values
    X=df.drop(columns=[c for c in ['DEFAULT','VIP','AGE_YEARS'] if c in df.columns]).astype(float).reset_index(drop=True); X=X.fillna(X.median(numeric_only=True)); return X,y,g
def adult(attr):
    df=pd.read_parquet(f"{D}/adult_income.parquet").sample(20000,random_state=0); y=(df.HIGH_INCOME==0).astype(int).values
    g=(df.sex.astype(str).str.strip().str.lower()=='female').astype(int).values if attr=='sex' else (df.race.astype(str).str.strip()!='White').astype(int).values
    raw=df.drop(columns=[c for c in ['HIGH_INCOME','income','sex','race','SEX_STD','AGE_YEARS','EDU_STD','VIP','fnlwgt'] if c in df.columns])
    cat=[c for c in raw.select_dtypes(include=['object','string']).columns]
    X=pd.get_dummies(raw,columns=cat).astype(float).reset_index(drop=True); return X,y,g
def compas():
    df=pd.read_csv(f"{D}/compas_two_years.csv")
    df=df[(df.days_b_screening_arrest<=30)&(df.days_b_screening_arrest>=-30)&(df.is_recid!=-1)&(df.c_charge_degree!='O')&(df.score_text!='N/A')]
    df=df[df.race.isin(['African-American','Caucasian'])]; y=df.two_year_recid.astype(int).values; g=(df.race=='African-American').astype(int).values
    raw=df[['age','priors_count','juv_fel_count','juv_misd_count','juv_other_count','c_charge_degree','sex']].copy()
    cat=[c for c in raw.select_dtypes(include=['object','string']).columns]
    X=pd.get_dummies(raw,columns=cat).astype(float).reset_index(drop=True); return X,y,g

SETS=[("german/age",*german()),("taiwan/sex",*taiwan('sex')),("taiwan/age",*taiwan('age')),
      ("taiwan/marital",*taiwan('marital')),("gmsc/age",*gmsc()),
      ("adult/sex",*adult('sex')),("adult/race",*adult('race')),("compas/race",*compas())]
SEEDS=list(range(10))
print(f"{'data/attr':16s} {'leak_logit':>10s} {'leak_gbm':>9s} | {'DIR_logit':>9s} {'DIR_gbm':>8s} {'AUC_gbm':>8s}")
rows=[]
for name,X,y,g in SETS:
    ll=leak_logit(X,g); lg=leak_gbm(X,g)
    dl=np.mean([base_dir(X,y,g,s,'logit')[1] for s in SEEDS])
    arr=np.array([base_dir(X,y,g,s,'gbm') for s in SEEDS]); dg=arr[:,1].mean(); ag=arr[:,0].mean()
    rows.append((name,ll,lg,dl,dg,ag))
    print(f"{name:16s} {ll:10.3f} {lg:9.3f} | {dl:9.3f} {dg:8.3f} {ag:8.3f}")
pd.DataFrame(rows,columns=['data_attr','leak_logit','leak_gbm','DIR_logit','DIR_gbm','AUC_gbm']).to_csv(
    os.path.join(_R,"results","tables","exp_robustness_leakage.csv"),index=False)

# ---- Figure: logit vs GBM leakage per dataset/attr ----
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid"); GRAYS=["#000000","#555555","#999999","#cccccc"]
plt.rcParams.update({"figure.dpi":600,"savefig.dpi":600,"font.size":11,
    "axes.edgecolor":"black","axes.linewidth":0.8,"grid.color":"0.85"})
dfp=pd.DataFrame(rows,columns=["data_attr","leak_logit","leak_gbm","DIR_logit","DIR_gbm","AUC_gbm"])
FIGD=os.path.join(_R,"results","figures")
fig,ax=plt.subplots(figsize=(8,4.4)); x=np.arange(len(dfp))
ax.bar(x-0.2,dfp["leak_logit"],0.4,label="logistic reconstructor",color=GRAYS[0],edgecolor="black",linewidth=1.0)
ax.bar(x+0.2,dfp["leak_gbm"],0.4,label="GBM reconstructor",color=GRAYS[2],edgecolor="black",linewidth=1.0)
ax.axhline(0.5,color="black",linestyle="--",linewidth=0.9)
ax.set_xticks(x); ax.set_xticklabels(dfp["data_attr"],rotation=30,ha="right")
ax.set_ylabel("Proxy leakage (reconstruction AUC)"); ax.set_ylim(0.4,1.0); ax.legend(frameon=True,edgecolor="black")
fig.tight_layout()
for ext in ("png","pdf"): fig.savefig(f"{FIGD}/exp8_leakage_logit_vs_gbm.{ext}",dpi=600,bbox_inches="tight")
plt.close(fig)
print("Done: robustness (strong leakage + GBM base).")


data/attr        leak_logit  leak_gbm | DIR_logit  DIR_gbm  AUC_gbm


german/age            0.800     0.780 |     0.828    0.886    0.770


taiwan/sex            0.546     0.602 |     1.013    1.021    0.779


taiwan/age            0.677     0.694 |     0.976    0.966    0.779


taiwan/marital        0.631     0.642 |     1.001    1.006    0.779


gmsc/age              0.695     0.739 |     0.996    0.985    0.846


adult/sex             0.928     0.931 |     0.316    0.325    0.923


adult/race            0.751     0.762 |     0.635    0.645    0.923


compas/race           0.686     0.677 |     0.675    0.689    0.720


Done: robustness (strong leakage + GBM base).
